## Setup

In [1]:
import yaml
import pandas as pd
import glob
from pathlib import Path
import json
import os
import sys
import jax.numpy as jnp
sys.path.append("..")

import wandb
api = wandb.Api()

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


In [2]:
root_sweeps = Path("../../wandb-sweep-data-qnrl/")
root_sweeps.mkdir(parents=True, exist_ok=True)

In [3]:
entity_id = "news-vt" # Team name in WandB.
project_id = "qnrl" # Project ID in WandB.

In [4]:
# List of sweep names and their IDs for reference.
sweep_name_id = { # name: id
    # QnRL.
    'qnrl/FrozenLake-v1': '7yoeyqqr',
    'qnrl/CliffWalking-v1': 'nin93kq8',
    'qnrl/Acrobot-v1': '5ehazde3',
    'qnrl/CartPole-v1': 'qojis1fw',
    'qnrl/Breakout-v5': 'eigl7wk5',
    'qnrl/SpaceInvaders-v5': '8adr9d1k',

    # C51.
    'c51/FrozenLake-v1': 'jzduwzvg',
    'c51/CliffWalking-v1': '8xyt1eb5',
    'c51/Acrobot-v1': 'n8imxz1v',
    'c51/CartPole-v1': 'arv708xs',
    'c51/Breakout-v5': 'dg4sodfc',
    'c51/SpaceInvaders-v5': 'wbmr06xp',
    
    # qC51.
    'qc51/FrozenLake-v1': 'n93n0nr1',
    'qc51/CliffWalking-v1': 'gicedw73',
    'qc51/Acrobot-v1': 'h0z9k7g5',
    'qc51/CartPole-v1': 'foiafm3w',
    'qc51/Breakout-v5': 'sya2ty4z',
    'qc51/SpaceInvaders-v5': 'vapn48ds',
}

In [5]:
metadata_path_tree = {
    # Each sweep has a dictionary of runs.
    Path(sweep_dir).stem.split('sweep-')[1]: {
        # Each run has a metadata JSON file.
        Path(run_dir).stem.split('run-')[1]: glob.glob(str(Path(run_dir)/'wandb-metadata.json')) for run_dir in glob.glob(str(Path(sweep_dir)/'*'))
    } for sweep_dir in glob.glob(str(root_sweeps/'*'))
}

In [6]:
summary_path_tree = {
    # Each sweep has a dictionary of runs.
    Path(sweep_dir).stem.split('sweep-')[1]: {
        # Each run has a metadata JSON file.
        Path(run_dir).stem.split('run-')[1]: glob.glob(str(Path(run_dir)/'wandb-summary.json')) for run_dir in glob.glob(str(Path(sweep_dir)/'*'))
    } for sweep_dir in glob.glob(str(root_sweeps/'*'))
}

In [7]:
config_path_tree = {
    # Each sweep has a dictionary of runs.
    Path(sweep_dir).stem.split('sweep-')[1]: {
        # Each run has a config YAML file.
        Path(run_dir).stem.split('run-')[1]: glob.glob(str(Path(run_dir)/'config.yaml')) for run_dir in glob.glob(str(Path(sweep_dir)/'*'))
    } for sweep_dir in glob.glob(str(root_sweeps/'*'))
}

In [8]:
history_path_tree = {
    # Each sweep has a dictionary of runs.
    Path(sweep_dir).stem.split('sweep-')[1]: {
        # Each run has a list of parquet files.
        Path(run_dir).stem.split('run-')[1]: glob.glob(str(Path(run_dir)/'history'/'*.parquet')) for run_dir in glob.glob(str(Path(sweep_dir)/'*'))
    } for sweep_dir in glob.glob(str(root_sweeps/'*'))
}

In [9]:
from collections import defaultdict
def group_runs_by_seed_group_hash(config_run_tree):
    seedgroups_tree = defaultdict(list)
    for run_id, config_paths in config_run_tree.items():
        if config_paths:
            with open(str(config_paths[0]), 'r') as f:
                config = yaml.safe_load(f)
            seed_group_hash = config["seed_group_hash"]["value"]
            seedgroups_tree[seed_group_hash].append(run_id)
    return dict(seedgroups_tree) # Convert to regular dictionary.

seedgroups_tree = {sweep_id: group_runs_by_seed_group_hash(run_tree) for sweep_id, run_tree in config_path_tree.items()}

In [10]:
from functools import reduce
def get_nested_value(d, path):
    return reduce(lambda x,y: x[y], path.split('.'),d)

agg_by_key = {
    'mean': lambda x: jnp.asarray(x).mean(),
    'std': lambda x: jnp.asarray(x).std(ddof=0),
}

def build_table_for_env(env_id: str, metrics_to_track: list[str], configs_to_track: list[str]):
    seed_group_metrics = {}
    seed_group_configs = {}
    for seed_group_hash, run_ids in seedgroups_tree[sweep_name_id[env_id]].items():
        seed_group_hash_stub = seed_group_hash.split('-', 1)[0]
        
        metric_keys_to_subagg = dict()
        
        metrics = defaultdict(list)
        configs = defaultdict(list)
        for run_id in run_ids:
            summary_path = summary_path_tree[sweep_name_id[env_id]][run_id][0]
            with open(summary_path, 'r') as f:
                summary = json.load(f)
            for metric_key in metrics_to_track:
                metric_stem, metric_agg = metric_key.split('.', 1)
                
                # Determine sub-aggregation method across seeds for metric via 'metric.agg:subagg', supports ('mean', 'std'), defaults to 'mean' if none is given.
                metric_agg, metric_subagg = (metric_agg.split(':', 1) + ['mean'])[:2]
                if metric_subagg in ('mean', 'std'):
                    metric_keys_to_subagg[metric_key] = metric_subagg
                else:
                    raise ValueError(f"Unsupported sub-aggregation {metric_subagg}")
                
                metrics[metric_key].append(summary[metric_stem][metric_agg])
                
            config_path = config_path_tree[sweep_name_id[env_id]][run_id][0]
            with open(config_path, 'r') as f:
                config = yaml.safe_load(f)
            for config_key in configs_to_track:
                # configs[config_key].append(config['config']['value'][config_key])
                configs[config_key].append(get_nested_value(config['config']['value'], config_key))
        
        # seed_group_metrics[seed_group_hash_stub] = {key: jnp.array(vals).mean() for key, vals in metrics.items()} # Aggregate the mean of all metrics for a seed group.
        seed_group_metrics[seed_group_hash_stub] = {key: agg_by_key[metric_keys_to_subagg[key]](vals) for key, vals in metrics.items()} # Aggregate the mean of all metrics for a seed group.
        seed_group_configs[seed_group_hash_stub] = {key: vals[0] for key, vals in configs.items()} # Just take the first one since they are all the same for a seed group.
    return seed_group_metrics, seed_group_configs

## C51

In [11]:
def make_table_c51_classic_control_and_toy_text(env_id: str):
    metrics_to_track = ['eval/episode_reward.mean', 'eval/episode_reward.mean:std', 'eval/total_model_parameter_count.mean']
    configs_to_track = ['model.din', 'model.num_actions', 'model.num_atoms', 'z_min', 'z_max', 'model.douts']
    seed_group_metrics, seed_group_configs = build_table_for_env(env_id, metrics_to_track, configs_to_track)
    # Format as dataframe.
    df = pd.concat([pd.DataFrame(seed_group_metrics).T, pd.DataFrame(seed_group_configs).T], axis=1)
    df = df.sort_index(ascending=True) # Sort by seed group hash stub.

    return (
        df.style
        .format({'eval/total_model_parameter_count.mean': lambda x: int(x)})
    )

def make_table_c51_atari(env_id: str):
    metrics_to_track = ['eval/episode_reward.mean', 'eval/episode_reward.mean:std', 'eval/total_model_parameter_count.mean']
    configs_to_track = ['model.din', 'model.frame_stack', 'model.w', 'model.h', 'model.num_actions', 'model.num_atoms', 'z_min', 'z_max', 'model.conv_douts', 'model.conv_kernel_size', 'model.conv_strides']
    seed_group_metrics, seed_group_configs = build_table_for_env(env_id, metrics_to_track, configs_to_track)
    # Format as dataframe.
    df = pd.concat([pd.DataFrame(seed_group_metrics).T, pd.DataFrame(seed_group_configs).T], axis=1)
    df = df.sort_index(ascending=True) # Sort by seed group hash stub.

    return (
        df.style
        .format({
            'eval/total_model_parameter_count.mean': lambda x: int(x),
            'model.conv_kernel_size': lambda x: jnp.array(x)[:,0].tolist(),
            'model.conv_strides': lambda x: jnp.array(x)[:,0].tolist(),
        })
    )

def make_table_c51(env_id: str):
    match env_id:
        case 'c51/CartPole-v1':
            styler = make_table_c51_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.3f}",
                'eval/episode_reward.mean:std': "{:.3f}",
            })
        case 'c51/CliffWalking-v1':
            styler = make_table_c51_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.1f}",
                'eval/episode_reward.mean:std': "{:.1f}",
            })
        case 'c51/FrozenLake-v1':
            styler = make_table_c51_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.4f}",
                'eval/episode_reward.mean:std': "{:.4f}",
            })
        case 'c51/Acrobot-v1':
            styler = make_table_c51_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.4f}",
                'eval/episode_reward.mean:std': "{:.4f}",
            })
        case 'c51/Breakout-v5':
            styler = make_table_c51_atari(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.2f}",
                'eval/episode_reward.mean:std': "{:.2f}",
            })
        case 'c51/SpaceInvaders-v5':
            styler = make_table_c51_atari(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.2f}",
                'eval/episode_reward.mean:std': "{:.2f}",
            })
        case _:
            raise ValueError(f"unsupported environment id {env_id}")

In [12]:
make_table_c51(env_id='c51/CartPole-v1')

,eval/episode_reward.mean,eval/episode_reward.mean:std,eval/total_model_parameter_count.mean,model.din,model.num_actions,model.num_atoms,z_min,z_max,model.douts
4368ff55,224.600,136.079,2209.0,4,2,32,-100,100,"[30, 21]"
49e46a87,469.300,18.585,16204.0,4,2,32,-100,100,"[120, 84]"
b7aea040,270.150,104.336,792.0,4,2,32,-100,100,"[16, 8]"
bdc26ac3,387.225,90.240,5614.0,4,2,32,-100,100,"[60, 42]"


In [13]:
make_table_c51(env_id='c51/CliffWalking-v1')

,eval/episode_reward.mean,eval/episode_reward.mean:std,eval/total_model_parameter_count.mean,model.din,model.num_actions,model.num_atoms,z_min,z_max,model.douts
15173423,-13.0,0.0,2072.0,48,4,32,-100,100,"[16, 8]"
8036b0d0,-77.5,37.2,590.0,48,4,32,-100,100,"[4, 2]"
8ef48998,-13.0,0.0,4937.0,48,4,32,-100,100,"[30, 21]"
b12ce064,-13.5,0.9,1151.0,48,4,32,-100,100,"[7, 5]"


In [14]:
make_table_c51(env_id='c51/FrozenLake-v1')

,eval/episode_reward.mean,eval/episode_reward.mean:std,eval/total_model_parameter_count.mean,model.din,model.num_actions,model.num_atoms,z_min,z_max,model.douts
03b4d6ac,-0.1708,0.0716,927.0,16,4,32,-2,2,"[7, 5]"
1c44782b,0.1518,0.1330,1560.0,16,4,32,-2,2,"[16, 8]"
a6bb645e,0.0925,0.1001,3977.0,16,4,32,-2,2,"[30, 21]"
d9663ea7,-0.3035,0.0986,462.0,16,4,32,-2,2,"[4, 2]"


In [15]:
make_table_c51(env_id='c51/Acrobot-v1')

,eval/episode_reward.mean,eval/episode_reward.mean:std,eval/total_model_parameter_count.mean,model.din,model.num_actions,model.num_atoms,z_min,z_max,model.douts
9f7469c2,-500.0000,0.0000,1112.0,6,3,32,-100,100,"[16, 8]"
c3881283,-320.9000,150.5338,7110.0,6,3,32,-100,100,"[60, 42]"
d615efcf,-363.3500,139.4894,2973.0,6,3,32,-100,100,"[30, 21]"
db125df2,-234.0750,154.3195,19164.0,6,3,32,-100,100,"[120, 84]"


In [16]:
make_table_c51(env_id='c51/SpaceInvaders-v5')

,eval/episode_reward.mean,eval/episode_reward.mean:std,eval/total_model_parameter_count.mean,model.din,model.frame_stack,model.w,model.h,model.num_actions,model.num_atoms,z_min,z_max,model.conv_douts,model.conv_kernel_size,model.conv_strides
1819f2cb,131.50,50.41,680288.0,1,4,84,84,6,32,-10,10,"[32, 64, 64]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
47ffe1c4,79.25,15.77,157160.0,1,4,84,84,6,32,-10,10,"[8, 16, 16]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
6feb00c2,71.25,45.42,322832.0,1,4,84,84,6,32,-10,10,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
f758df04,142.25,30.53,1.499648e+06,1,4,84,84,6,32,-10,10,"[64, 128, 128]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"


In [17]:
make_table_c51(env_id='c51/Breakout-v5')

,eval/episode_reward.mean,eval/episode_reward.mean:std,eval/total_model_parameter_count.mean,model.din,model.frame_stack,model.w,model.h,model.num_actions,model.num_atoms,z_min,z_max,model.conv_douts,model.conv_kernel_size,model.conv_strides
33e43f68,3.55,1.52,222416.0,1,4,84,84,4,32,-10,10,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
99dbcbf7,1.80,0.93,106920.0,1,4,84,84,4,32,-10,10,"[8, 16, 16]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
9c1e6421,5.20,1.22,1.098176e+06,1,4,84,84,4,32,-10,10,"[64, 128, 128]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
beb2a303,3.55,1.02,479520.0,1,4,84,84,4,32,-10,10,"[32, 64, 64]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"


## qC51

In [18]:
def make_table_qc51_classic_control_and_toy_text(env_id: str):
    metrics_to_track = ['eval/episode_reward.mean', 'eval/episode_reward.mean:std', 'eval/total_model_parameter_count.mean']
    configs_to_track = ['model.din', 'model.num_actions', 'model.num_atoms', 'model.num_wires', 'model.num_layers', 'z_min', 'z_max']
    seed_group_metrics, seed_group_configs = build_table_for_env(env_id, metrics_to_track, configs_to_track)
    # Format as dataframe.
    df = pd.concat([pd.DataFrame(seed_group_metrics).T, pd.DataFrame(seed_group_configs).T], axis=1)
    df = df.sort_index(ascending=True) # Sort by seed group hash stub.

    return (
        df.style
        .format({'eval/total_model_parameter_count.mean': lambda x: int(x)})
    )

def make_table_qc51_atari(env_id: str):
    metrics_to_track = ['eval/episode_reward.mean', 'eval/episode_reward.mean:std', 'eval/total_model_parameter_count.mean']
    configs_to_track = ['model.din', 'model.frame_stack', 'model.w', 'model.h', 'model.num_actions', 'model.num_atoms', 'model.num_wires', 'model.num_layers', 'z_min', 'z_max', 'model.conv_douts', 'model.conv_kernel_size', 'model.conv_strides']
    seed_group_metrics, seed_group_configs = build_table_for_env(env_id, metrics_to_track, configs_to_track)
    # Format as dataframe.
    df = pd.concat([pd.DataFrame(seed_group_metrics).T, pd.DataFrame(seed_group_configs).T], axis=1)
    df = df.sort_index(ascending=True) # Sort by seed group hash stub.

    return (
        df.style
        .format({
            'eval/total_model_parameter_count.mean': lambda x: int(x),
            'model.conv_kernel_size': lambda x: jnp.array(x)[:,0].tolist(),
            'model.conv_strides': lambda x: jnp.array(x)[:,0].tolist(),
        })
    )

def make_table_qc51(env_id: str):
    match env_id:
        case 'qc51/CartPole-v1':
            styler = make_table_qc51_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.3f}",
                'eval/episode_reward.mean:std': "{:.3f}",
            })
        case 'qc51/CliffWalking-v1':
            styler = make_table_qc51_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.1f}",
                'eval/episode_reward.mean:std': "{:.1f}",
            })
        case 'qc51/FrozenLake-v1':
            styler = make_table_qc51_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.4f}",
                'eval/episode_reward.mean:std': "{:.4f}",
            })
        case 'qc51/Acrobot-v1':
            styler = make_table_qc51_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.4f}",
                'eval/episode_reward.mean:std': "{:.4f}",
            })
        case 'qc51/Breakout-v5':
            styler = make_table_qc51_atari(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.2f}",
                'eval/episode_reward.mean:std': "{:.2f}",
            })
        case 'qc51/SpaceInvaders-v5':
            styler = make_table_qc51_atari(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.2f}",
                'eval/episode_reward.mean:std': "{:.2f}",
            })
        case _:
            raise ValueError(f"unsupported environment id {env_id}")

In [19]:
make_table_qc51(env_id='qc51/CartPole-v1')

,eval/episode_reward.mean,eval/episode_reward.mean:std,eval/total_model_parameter_count.mean,model.din,model.num_actions,model.num_atoms,model.num_wires,model.num_layers,z_min,z_max
05a20b51,263.950,150.158,433.0,4,2,32,3,7,-100,100
1a617f87,412.575,151.425,599.0,4,2,32,5,5,-100,100
47fccc78,420.275,138.088,925.0,4,2,32,7,7,-100,100
593724e1,186.450,21.572,519.0,4,2,32,5,3,-100,100
8676ecba,371.575,157.013,701.0,4,2,32,7,3,-100,100
d2a71ccc,184.275,67.482,679.0,4,2,32,5,7,-100,100
d44671c0,332.475,172.314,813.0,4,2,32,7,5,-100,100
d5d8fb91,176.800,82.382,385.0,4,2,32,3,5,-100,100
e36b7b7a,231.700,140.404,337.0,4,2,32,3,3,-100,100


In [20]:
make_table_qc51(env_id='qc51/CliffWalking-v1')

,eval/episode_reward.mean,eval/episode_reward.mean:std,eval/total_model_parameter_count.mean,model.din,model.num_actions,model.num_atoms,model.num_wires,model.num_layers,z_min,z_max
05dfa4e9,-13.0,0.0,1613.0,48,4,32,3,7,-100,100
23f90783,-13.0,0.0,2137.0,48,4,32,7,3,-100,100
542f7acf,-13.0,0.0,2603.0,48,4,32,5,7,-100,100
6034de7f,-13.0,0.0,1563.0,48,4,32,5,3,-100,100
77669710,-34.5,37.2,989.0,48,4,32,3,3,-100,100
b29dfe3e,-56.0,43.0,1301.0,48,4,32,3,5,-100,100
bb502e99,-13.0,0.0,2083.0,48,4,32,5,5,-100,100
cf760085,-13.0,0.0,3593.0,48,4,32,7,7,-100,100
f1f70b1d,-13.0,0.0,2865.0,48,4,32,7,5,-100,100


In [21]:
make_table_qc51(env_id='qc51/FrozenLake-v1')

,eval/episode_reward.mean,eval/episode_reward.mean:std,eval/total_model_parameter_count.mean,model.din,model.num_actions,model.num_atoms,model.num_wires,model.num_layers,z_min,z_max
2e1f1216,-0.2015,0.0702,821.0,16,4,32,3,5,-2,2
42168c4b,0.1840,0.0911,1745.0,16,4,32,7,5,-2,2
490888e6,0.0382,0.2408,1083.0,16,4,32,5,3,-2,2
67772df3,-0.1838,0.0656,941.0,16,4,32,3,7,-2,2
7e2656fe,-0.1277,0.1133,701.0,16,4,32,3,3,-2,2
864c90ba,0.1525,0.0849,2025.0,16,4,32,7,7,-2,2
b72490e5,0.0373,0.1320,1465.0,16,4,32,7,3,-2,2
c0445e6b,-0.0097,0.0245,1483.0,16,4,32,5,7,-2,2
c3eb5d77,-0.0422,0.1203,1283.0,16,4,32,5,5,-2,2


In [22]:
make_table_qc51(env_id='qc51/Acrobot-v1')

,eval/episode_reward.mean,eval/episode_reward.mean:std,eval/total_model_parameter_count.mean,model.din,model.num_actions,model.num_atoms,model.num_wires,model.num_layers,z_min,z_max
35c0b15f,-500.0000,0.0000,603.0,6,3,32,3,7,-100,100
45b8140e,-423.7000,132.1555,483.0,6,3,32,3,3,-100,100
55bba206,-449.9750,86.6458,1139.0,6,3,32,7,5,-100,100
658c80b9,-500.0000,0.0000,941.0,6,3,32,5,7,-100,100
7ba52505,-257.0000,171.3579,999.0,6,3,32,7,3,-100,100
84024276,-500.0000,0.0000,1279.0,6,3,32,7,7,-100,100
8d2966fb,-500.0000,0.0000,543.0,6,3,32,3,5,-100,100
ded7a44a,-479.9500,34.7276,841.0,6,3,32,5,5,-100,100
ff546b55,-420.8750,137.0485,741.0,6,3,32,5,3,-100,100


In [23]:
make_table_qc51(env_id='qc51/SpaceInvaders-v5')

,eval/episode_reward.mean,eval/episode_reward.mean:std,eval/total_model_parameter_count.mean,model.din,model.frame_stack,model.w,model.h,model.num_actions,model.num_atoms,model.num_wires,model.num_layers,z_min,z_max,model.conv_douts,model.conv_kernel_size,model.conv_strides
010b112a,106.00,25.25,125861.0,1,4,84,84,6,32,3,5,-10,10,"[32, 64, 64]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
08e92238,78.25,16.74,24545.0,1,4,84,84,6,32,7,3,-10,10,"[8, 16, 16]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
0bdf8416,106.00,35.05,36509.0,1,4,84,84,6,32,3,3,-10,10,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
15421747,81.50,25.50,19427.0,1,4,84,84,6,32,5,3,-10,10,"[8, 16, 16]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
18c74f45,76.25,33.42,35187.0,1,4,84,84,6,32,5,7,-10,10,"[8, 16, 16]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
192bfb4b,80.75,50.31,35577.0,1,4,84,84,6,32,7,5,-10,10,"[8, 16, 16]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
20047e6e,65.75,25.66,62051.0,1,4,84,84,6,32,5,5,-10,10,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
21809e01,121.75,59.87,427805.0,1,4,84,84,6,32,3,7,-10,10,"[64, 128, 128]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
34bb52d9,91.75,15.55,107021.0,1,4,84,84,6,32,3,3,-10,10,"[32, 64, 64]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
352ceed0,76.25,32.76,157651.0,1,4,84,84,6,32,5,5,-10,10,"[32, 64, 64]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"


In [24]:
make_table_qc51(env_id='qc51/Breakout-v5')

,eval/episode_reward.mean,eval/episode_reward.mean:std,eval/total_model_parameter_count.mean,model.din,model.frame_stack,model.w,model.h,model.num_actions,model.num_atoms,model.num_wires,model.num_layers,z_min,z_max,model.conv_douts,model.conv_kernel_size,model.conv_strides
015c0468,1.35,0.48,188667.0,1,4,84,84,4,32,5,7,-10,10,"[32, 64, 64]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
08d7245c,1.00,0.97,427549.0,1,4,84,84,4,32,3,7,-10,10,"[64, 128, 128]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
09df8179,1.65,1.05,144969.0,1,4,84,84,4,32,7,3,-10,10,"[32, 64, 64]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
0f7d455e,1.25,0.43,35065.0,1,4,84,84,4,32,7,5,-10,10,"[8, 16, 16]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
18881af9,0.60,0.51,36253.0,1,4,84,84,4,32,3,3,-10,10,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
1ba64303,1.10,1.24,428073.0,1,4,84,84,4,32,7,3,-10,10,"[64, 128, 128]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
1c99b831,2.45,1.25,188929.0,1,4,84,84,4,32,7,5,-10,10,"[32, 64, 64]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
22701ae4,3.20,2.38,515937.0,1,4,84,84,4,32,7,5,-10,10,"[64, 128, 128]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
28e6b8f8,0.27,0.25,55117.0,1,4,84,84,4,32,3,7,-10,10,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
2a718710,1.35,1.37,61667.0,1,4,84,84,4,32,5,5,-10,10,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"


## QnRL

In [25]:
def make_table_qnrl_classic_control_and_toy_text(env_id: str):
    metrics_to_track = ['eval/episode_reward.mean', 'eval/episode_reward.mean:std', 'eval/total_model_parameter_count.mean']
    configs_to_track = ['model.din', 'model.n_actions', 'model.n_returns', 'model.z_min', 'model.z_max', 'model.n_reuploads', 'model.m_moment', 'model.n_layers', 'model.pauli_enc', 'model.pauli_var', 'model.imprimitive', 'model.input_multiplex_activation']
    seed_group_metrics, seed_group_configs = build_table_for_env(env_id, metrics_to_track, configs_to_track)
    # Format as dataframe.
    df = pd.concat([pd.DataFrame(seed_group_metrics).T, pd.DataFrame(seed_group_configs).T], axis=1)
    df = df.sort_index(ascending=True) # Sort by seed group hash stub.

    return (
        df.style
        .format({'eval/total_model_parameter_count.mean': lambda x: int(x)})
    )

def make_table_qnrl_atari(env_id: str):
    metrics_to_track = ['eval/episode_reward.mean', 'eval/episode_reward.mean:std', 'eval/total_model_parameter_count.mean']
    configs_to_track = ['model.din', 'model.frame_stack', 'model.w', 'model.h', 'model.n_actions', 'model.n_returns', 'model.z_min', 'model.z_max', 'model.n_reuploads', 'model.m_moment', 'model.n_layers', 'model.pauli_enc', 'model.pauli_var', 'model.imprimitive', 'model.input_multiplex_activation', 'model.conv_douts', 'model.conv_kernel_size', 'model.conv_strides']
    seed_group_metrics, seed_group_configs = build_table_for_env(env_id, metrics_to_track, configs_to_track)
    # Format as dataframe.
    df = pd.concat([pd.DataFrame(seed_group_metrics).T, pd.DataFrame(seed_group_configs).T], axis=1)
    df = df.sort_index(ascending=True) # Sort by seed group hash stub.

    return (
        df.style
        .format({
            'eval/total_model_parameter_count.mean': lambda x: int(x),
            'model.conv_kernel_size': lambda x: jnp.array(x)[:,0].tolist(),
            'model.conv_strides': lambda x: jnp.array(x)[:,0].tolist(),
        })
    )

def make_table_qnrl(env_id: str):
    match env_id:
        case 'qnrl/CartPole-v1':
            styler = make_table_qnrl_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.3f}",
                'eval/episode_reward.mean:std': "{:.3f}",
            })
        case 'qnrl/CliffWalking-v1':
            styler = make_table_qnrl_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.1f}",
                'eval/episode_reward.mean:std': "{:.1f}",
            })
        case 'qnrl/FrozenLake-v1':
            styler = make_table_qnrl_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.4f}",
                'eval/episode_reward.mean:std': "{:.4f}",
            })
        case 'qnrl/Acrobot-v1':
            styler = make_table_qnrl_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.4f}",
                'eval/episode_reward.mean:std': "{:.4f}",
            })
        case 'qnrl/Breakout-v5':
            styler = make_table_qnrl_atari(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.2f}",
                'eval/episode_reward.mean:std': "{:.2f}",
            })
        case 'qnrl/SpaceInvaders-v5':
            styler = make_table_qnrl_atari(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.2f}",
                'eval/episode_reward.mean:std': "{:.2f}",
            })
        case _:
            raise ValueError(f"unsupported environment id {env_id}")

In [26]:
make_table_qnrl(env_id='qnrl/CartPole-v1')

,eval/episode_reward.mean,eval/episode_reward.mean:std,eval/total_model_parameter_count.mean,model.din,model.n_actions,model.n_returns,model.z_min,model.z_max,model.n_reuploads,model.m_moment,model.n_layers,model.pauli_enc,model.pauli_var,model.imprimitive,model.input_multiplex_activation
0a1d6db2,336.150,163.854,330.0,4,2,32,-100,100,2,2,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
14a05a3d,241.625,109.827,330.0,4,2,32,-100,100,2,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
171a91d1,336.150,163.854,330.0,4,2,32,-100,100,1,2,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
1f3b03a6,168.150,84.536,250.0,4,2,32,-100,100,2,2,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
1f8ffc6f,168.150,84.536,250.0,4,2,32,-100,100,1,2,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
31994c7a,241.625,109.827,330.0,4,2,32,-100,100,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
3a00cecd,256.475,132.158,250.0,4,2,32,-100,100,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
405bd6e3,43.025,30.692,170.0,4,2,32,-100,100,2,2,3,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
45913de1,452.300,72.225,330.0,4,2,32,-100,100,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
45b9d666,256.475,132.158,250.0,4,2,32,-100,100,2,1,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh


In [27]:
make_table_qnrl(env_id='qnrl/CliffWalking-v1')

,eval/episode_reward.mean,eval/episode_reward.mean:std,eval/total_model_parameter_count.mean,model.din,model.n_actions,model.n_returns,model.z_min,model.z_max,model.n_reuploads,model.m_moment,model.n_layers,model.pauli_enc,model.pauli_var,model.imprimitive,model.input_multiplex_activation
08d163cb,-34.5,37.2,1380.0,48,4,32,-100,100,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
0b5c3b76,-56.0,43.0,1380.0,48,4,32,-100,100,1,2,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
12c045e6,-13.0,0.0,1220.0,48,4,32,-100,100,1,1,3,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
1729a977,-56.0,43.0,1540.0,48,4,32,-100,100,1,2,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
1fc6943d,-35.0,37.0,1540.0,48,4,32,-100,100,4,1,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
2d18e0c2,-13.0,0.0,1220.0,48,4,32,-100,100,4,1,3,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
2f5083d1,-77.5,37.2,1220.0,48,4,32,-100,100,1,2,3,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
3992cb09,-13.0,0.0,1380.0,48,4,32,-100,100,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
3c16ef96,-13.0,0.0,1540.0,48,4,32,-100,100,1,2,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
773dd844,-34.5,37.2,1540.0,48,4,32,-100,100,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh


In [28]:
make_table_qnrl(env_id='qnrl/FrozenLake-v1')

,eval/episode_reward.mean,eval/episode_reward.mean:std,eval/total_model_parameter_count.mean,model.din,model.n_actions,model.n_returns,model.z_min,model.z_max,model.n_reuploads,model.m_moment,model.n_layers,model.pauli_enc,model.pauli_var,model.imprimitive,model.input_multiplex_activation
09fc2cfb,0.0990,0.1471,580.0,16,4,32,-2,2,4,2,3,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
0beeeeea,0.1623,0.0574,900.0,16,4,32,-2,2,1,2,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
1a6e23d3,0.0480,0.2172,580.0,16,4,32,-2,2,1,1,3,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
2a30337c,0.0990,0.1471,580.0,16,4,32,-2,2,1,2,3,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
2cae8b90,0.1815,0.0932,900.0,16,4,32,-2,2,4,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
35d94c95,0.1937,0.0437,740.0,16,4,32,-2,2,4,2,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
3cee6e98,0.2500,0.0208,740.0,16,4,32,-2,2,1,2,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
4270289a,0.2120,0.1882,740.0,16,4,32,-2,2,4,1,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
485c0bf8,0.1622,0.0574,900.0,16,4,32,-2,2,4,2,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
5064f085,0.2120,0.1882,740.0,16,4,32,-2,2,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh


In [29]:
make_table_qnrl(env_id='qnrl/Acrobot-v1')

,eval/episode_reward.mean,eval/episode_reward.mean:std,eval/total_model_parameter_count.mean,model.din,model.n_actions,model.n_returns,model.z_min,model.z_max,model.n_reuploads,model.m_moment,model.n_layers,model.pauli_enc,model.pauli_var,model.imprimitive,model.input_multiplex_activation
01f1b2a8,-204.2750,171.2223,405.0,6,3,32,-100,100,3,1,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
03b176ff,-500.0000,0.0000,405.0,6,3,32,-100,100,3,2,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
043b80e2,-500.0000,0.0000,525.0,6,3,32,-100,100,3,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
0ff9fb8e,-500.0000,0.0000,285.0,6,3,32,-100,100,3,2,3,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
10d163eb,-415.7750,123.3515,405.0,6,3,32,-100,100,3,1,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
1bacc532,-196.4250,136.4100,405.0,6,3,32,-100,100,1,2,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
24689683,-419.9000,80.1427,525.0,6,3,32,-100,100,1,2,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
25417497,-415.7750,123.3515,405.0,6,3,32,-100,100,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
283bce61,-428.6750,123.5385,285.0,6,3,32,-100,100,3,2,3,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
43229ab1,-500.0000,0.0000,525.0,6,3,32,-100,100,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh


In [30]:
make_table_qnrl(env_id='qnrl/SpaceInvaders-v5')

,eval/episode_reward.mean,eval/episode_reward.mean:std,eval/total_model_parameter_count.mean,model.din,model.frame_stack,model.w,model.h,model.n_actions,model.n_returns,model.z_min,model.z_max,model.n_reuploads,model.m_moment,model.n_layers,model.pauli_enc,model.pauli_var,model.imprimitive,model.input_multiplex_activation,model.conv_douts,model.conv_kernel_size,model.conv_strides
28cf3201,91.50,37.82,69494.0,1,4,84,84,6,32,-10,10,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
2a976698,88.75,18.06,69494.0,1,4,84,84,6,32,-10,10,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
3f1e580c,84.75,18.83,172694.0,1,4,84,84,6,32,-10,10,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh,"[32, 64, 64]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
4031053e,49.50,27.68,172934.0,1,4,84,84,6,32,-10,10,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh,"[32, 64, 64]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
4801d71d,99.25,56.75,69254.0,1,4,84,84,6,32,-10,10,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
56217398,83.25,17.61,484262.0,1,4,84,84,6,32,-10,10,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[64, 128, 128]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
5a7a5c51,97.25,21.03,484022.0,1,4,84,84,6,32,-10,10,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[64, 128, 128]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
70aae234,73.50,36.73,30830.0,1,4,84,84,6,32,-10,10,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh,"[8, 16, 16]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
7fbdd5b1,53.50,38.41,172934.0,1,4,84,84,6,32,-10,10,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[32, 64, 64]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
8066810a,154.50,79.76,484262.0,1,4,84,84,6,32,-10,10,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh,"[64, 128, 128]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"


In [31]:
make_table_qnrl(env_id='qnrl/Breakout-v5')

,eval/episode_reward.mean,eval/episode_reward.mean:std,eval/total_model_parameter_count.mean,model.din,model.frame_stack,model.w,model.h,model.n_actions,model.n_returns,model.z_min,model.z_max,model.n_reuploads,model.m_moment,model.n_layers,model.pauli_enc,model.pauli_var,model.imprimitive,model.input_multiplex_activation,model.conv_douts,model.conv_kernel_size,model.conv_strides
05815f17,1.10,0.81,22700.0,1,4,84,84,4,32,-10,10,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[8, 16, 16]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
069ad0e5,1.95,0.50,53204.0,1,4,84,84,4,32,-10,10,1,1,3,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
1a8fc558,2.75,0.36,141284.0,1,4,84,84,4,32,-10,10,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[32, 64, 64]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
2153e7e5,1.30,1.14,22380.0,1,4,84,84,4,32,-10,10,1,1,3,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh,"[8, 16, 16]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
21892ef5,0.40,0.32,22380.0,1,4,84,84,4,32,-10,10,1,1,3,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[8, 16, 16]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
48b698e2,3.15,1.27,421092.0,1,4,84,84,4,32,-10,10,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[64, 128, 128]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
49fee01c,1.85,1.21,53204.0,1,4,84,84,4,32,-10,10,1,1,3,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
4d2222f7,4.90,1.87,421092.0,1,4,84,84,4,32,-10,10,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh,"[64, 128, 128]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
4dbdc680,0.85,0.57,53364.0,1,4,84,84,4,32,-10,10,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
55bfe7f3,2.40,0.84,53364.0,1,4,84,84,4,32,-10,10,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
